# Aula 12 - Deploy de modelo e criação de pipeline de processamento

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**29/09/2026 - Sprint 5 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula12.ipynb)

## O que este notebook é

A Aula 11 terminou com um vencedor: regressão linear sobre o alvo em nível, que erra **2,86%** nos
24 meses de teste. Este notebook pega esse vencedor e o prepara para sair do notebook, que é o que
a Sprint 5 pede.

São três coisas, e cada uma responde a uma pergunta:

1. **`Pipeline`**: o que precisa viajar junto com o modelo? O escalador, e na ordem certa.
2. **Export com `joblib`**: o modelo sobrevive a ser gravado em disco e recarregado em outra
   máquina, com outra versão de biblioteca?
3. **MLflow**: daqui a três meses, alguém consegue dizer de onde veio este número?

No meio do caminho aparece o achado da aula, e ele não estava no roteiro: **reajustar o modelo
sobre a base completa move os coeficientes numa mediana de 17,2%, faz um deles trocar de sinal, e
quase não move a previsão.** O que isso significa para o que a dupla entrega está na seção 4.

Tudo aqui roda sobre a mesma base mensal das Aulas 07 a 11: 339 linhas, 315 meses de treino, 24 de
teste, onze features.

## 1. A base e o modelo que a Aula 11 escolheu

A célula abaixo remonta a base analítica e reajusta o vencedor da Aula 11, para o notebook ficar
autocontido. Nada aqui é novo: é o ponto de partida.

In [1]:
import calendar
import os
import subprocess
import sys
import urllib.request
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"
FEATURES = (["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
            + [serie + "_lag1" for serie in SERIES if serie != ALVO])
N_TESTE = 24

MENSAL_LOCAL = os.path.join("..", "dados", "mensal")
MENSAL_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
                "main/dados/mensal/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(MENSAL_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(MENSAL_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos[nome] = arquivo

base = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)

base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)
base = base.dropna().reset_index(drop=True)

corte = len(base) - N_TESTE
treino, teste = base.iloc[:corte], base.iloc[corte:]

X_tr, y_tr = treino[FEATURES].to_numpy(), treino[ALVO].to_numpy()
X_te, y_te = teste[FEATURES].to_numpy(), teste[ALVO].to_numpy()
X_full, y_full = base[FEATURES].to_numpy(), base[ALVO].to_numpy()


def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


print("base  : %d linhas, de %s a %s" % (len(base), base["periodo"].iloc[0],
                                         base["periodo"].iloc[-1]))
print("treino: %d meses, ate %s" % (len(treino), treino["periodo"].iloc[-1]))
print("teste : %d meses, de %s a %s" % (len(teste), teste["periodo"].iloc[0],
                                        teste["periodo"].iloc[-1]))

base  : 339 linhas, de 1998-01 a 2026-03
treino: 315 meses, ate 2024-03
teste : 24 meses, de 2024-04 a 2026-03


## 2. O `Pipeline` não muda número nenhum

A Aula 11 fazia isto em dois objetos separados: um `StandardScaler` ajustado no treino, e uma
`LinearRegression` ajustada sobre o resultado dele. Funciona, e tem um problema: a ordem certa
existe só na cabeça de quem escreveu o código.

O `Pipeline` encapsula os dois num objeto. A célula abaixo faz as duas coisas e compara, porque a
primeira pergunta razoável é se encapsular muda alguma coisa.

In [2]:
# o passo a passo da Aula 11
escalador = StandardScaler().fit(X_tr)
modelo = LinearRegression().fit(escalador.transform(X_tr), y_tr)
passo_a_passo = modelo.predict(escalador.transform(X_te))

# o mesmo, dentro de um Pipeline
pipe = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", LinearRegression()),
])
pipe.fit(X_tr, y_tr)
pelo_pipeline = pipe.predict(X_te)

print("MAPE do passo a passo: %.10f%%" % mape(y_te, passo_a_passo))
print("MAPE do Pipeline     : %.10f%%" % mape(y_te, pelo_pipeline))
print()
print("identico bit a bit:", np.array_equal(passo_a_passo, pelo_pipeline))
print("maior diferenca absoluta entre as 24 previsoes: %.1e"
      % np.max(np.abs(passo_a_passo - pelo_pipeline)))

MAPE do passo a passo: 2.8620136234%
MAPE do Pipeline     : 2.8620136234%

identico bit a bit: True
maior diferenca absoluta entre as 24 previsoes: 0.0e+00


O resultado é idêntico bit a bit. **O `Pipeline` não é uma técnica de melhorar modelo**, e quem
esperava ganho de precisão aqui não vai encontrar.

O que ele dá é outra coisa, e ela aparece quando o modelo entra num validador que reparte o dado.
Na Aula 10, o `TimeSeriesSplit` treina cinco vezes em recortes diferentes. Com escalador solto,
cada uma dessas cinco vezes precisa reajustar o escalador na dobra certa, e é responsabilidade de
quem escreve o laço lembrar disso. Com o `Pipeline`, o `cross_val_score` reajusta o objeto inteiro
por dobra, e a ordem deixa de depender de memória.

A célula abaixo mostra o que o objeto carrega dentro.

In [3]:
print(pipe)
print()
print("passos:", list(pipe.named_steps))
print("media do escalador (3 primeiras features):",
      np.round(pipe.named_steps["escala"].mean_[:3], 2))
print("coeficientes do modelo (3 primeiros)     :",
      np.round(pipe.named_steps["modelo"].coef_[:3], 2))
print()
print("O que viaja junto com o modelo: %d medias, %d desvios e %d coeficientes."
      % (len(pipe.named_steps["escala"].mean_),
         len(pipe.named_steps["escala"].scale_),
         len(pipe.named_steps["modelo"].coef_)))

Pipeline(steps=[('escala', StandardScaler()), ('modelo', LinearRegression())])

passos: ['escala', 'modelo']
media do escalador (3 primeiras features): [7.48905203e+08 7.46335317e+08 7.43691402e+08]
coeficientes do modelo (3 primeiros)     : [11058622.23 86205775.92 93695988.77]

O que viaja junto com o modelo: 11 medias, 11 desvios e 11 coeficientes.


## 3. O modelo que vai para produção não é o modelo que foi avaliado

Aqui a aula sai do roteiro previsível, e é o ponto que a dupla precisa entender antes de exportar
qualquer coisa.

O modelo da seção 2 treinou com 315 meses e foi medido nos 24 que ele nunca viu. Esse é o
protocolo honesto, e o **2,86% é o número que descreve o modelo**. Mas agora a avaliação terminou:
não há mais nada a decidir, e deixar 24 meses de dado real parados fora do modelo que vai para a
LDC seria desperdício.

Então o objeto exportado é reajustado sobre as 339 linhas. A célula abaixo faz isso e compara os
dois.

In [4]:
producao = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", LinearRegression()),
]).fit(X_full, y_full)

coef_avaliado = pipe.named_steps["modelo"].coef_
coef_producao = producao.named_steps["modelo"].coef_
desloc = np.abs((coef_producao - coef_avaliado) / coef_avaliado) * 100

quadro = pd.DataFrame({
    "feature": FEATURES,
    "coef_avaliado": np.round(coef_avaliado, 1),
    "coef_producao": np.round(coef_producao, 1),
    "deslocamento_%": np.round(desloc, 1),
}).sort_values("deslocamento_%", ascending=False)
print(quadro.to_string(index=False))
print()
print("deslocamento mediano: %.1f%%" % np.median(desloc))
print("trocaram de sinal   : %s"
      % [f for f, a, b in zip(FEATURES, coef_avaliado, coef_producao)
         if np.sign(a) != np.sign(b)])

            feature  coef_avaliado  coef_producao  deslocamento_%
 abate_bovinos_lag1       -93403.7      3118713.9          3439.0
 producao_ovos_lag1      8186585.6     15310656.3            87.0
  abate_suinos_lag1     -9717140.4    -15903851.0            63.7
producao_leite_lag1      7640083.7      4005305.2            47.6
               lag1     11058622.2      6298310.1            43.0
                sen      5419254.9      6351251.7            17.2
               lag3     93695988.8    105597973.5            12.7
              lag12     47561974.0     52698456.9            10.8
                cos     -7351456.6     -6909108.8             6.0
               lag2     86205775.9     90079110.0             4.5
               dias     14865891.3     14735866.2             0.9

deslocamento mediano: 17.2%
trocaram de sinal   : ['abate_bovinos_lag1']


Vinte e quatro meses a mais, e os coeficientes se movem numa mediana de 17,2%. O de
`abate_bovinos_lag1` **troca de sinal**: o modelo avaliado dizia que abate de bovinos no mês
anterior puxa a produção de frango para baixo, e o de produção diz que puxa para cima.

A pergunta óbvia é quanto isso estraga a previsão. A resposta está na célula abaixo, e é o
contrário do que a tabela acima sugere.

In [5]:
prev_avaliado = pipe.predict(X_te)
prev_producao = producao.predict(X_te)
desloc_prev = np.abs((prev_producao - prev_avaliado) / prev_avaliado) * 100

print("deslocamento das previsoes nos mesmos 24 meses")
print("  media : %.4f%%" % np.mean(desloc_prev))
print("  maximo: %.4f%%" % np.max(desloc_prev))
print()
print("coeficiente move mediana de %.1f%%, previsao move no maximo %.2f%%."
      % (np.median(desloc), np.max(desloc_prev)))
print("A razao entre as duas grandezas e de cerca de %d vezes."
      % round(np.median(desloc) / np.mean(desloc_prev)))

deslocamento das previsoes nos mesmos 24 meses
  media : 0.6465%
  maximo: 0.9724%

coeficiente move mediana de 17.2%, previsao move no maximo 0.97%.
A razao entre as duas grandezas e de cerca de 27 vezes.


### Por que isso acontece

As onze features são fortemente correlacionadas entre si: `lag1`, `lag2`, `lag3` e `lag12` são a
mesma série deslocada, e as quatro séries de outras proteínas crescem junto com a de frango. Quando
as entradas são colineares, existem muitas combinações de coeficientes que produzem quase o mesmo
ajuste, e o modelo troca livremente peso entre elas. A célula abaixo mede isso.

In [6]:
Z = StandardScaler().fit_transform(X_tr)
C = np.corrcoef(Z, rowvar=False)

pares = [(FEATURES[i], FEATURES[j], C[i, j])
         for i in range(len(FEATURES))
         for j in range(i + 1, len(FEATURES))
         if abs(C[i, j]) > 0.90]
print("pares de features com correlacao acima de 0,90: %d" % len(pares))
for a, b, r in pares[:5]:
    print("  %-20s %-20s %+.3f" % (a, b, r))
print("  ...")
print()

vif = np.diag(np.linalg.inv(C))
print("VIF (fator de inflacao da variancia), acima de 10 e o limiar usual:")
for nome, v in sorted(zip(FEATURES, vif), key=lambda t: -t[1]):
    marca = "  <-" if v > 10 else ""
    print("  %-20s %6.1f%s" % (nome, v, marca))

pares de features com correlacao acima de 0,90: 20
  lag1                 lag2                 +0.970
  lag1                 lag3                 +0.981
  lag1                 lag12                +0.969
  lag1                 abate_bovinos_lag1   +0.909
  lag1                 abate_suinos_lag1    +0.965
  ...

VIF (fator de inflacao da variancia), acima de 10 e o limiar usual:
  lag1                   69.0  <-
  abate_suinos_lag1      47.2  <-
  lag3                   38.9  <-
  lag12                  36.8  <-
  lag2                   27.9  <-
  producao_ovos_lag1     23.9  <-
  producao_leite_lag1    23.4  <-
  abate_bovinos_lag1      7.4
  cos                     1.8
  dias                    1.6
  sen                     1.3


**Cuidado com uma confusão possível.** A Aula 05 mediu um número de condição de 1,08e10 nesta mesma
família de dados e resolveu o problema padronizando. Este aqui é outro problema, com a mesma
aparência: padronizar levou o número de condição de 3,2e9 para 28,5, e o VIF de `lag1` continua em
69. Condicionamento numérico e colinearidade estatística são coisas diferentes, e a segunda não se
resolve com escala.

A célula abaixo confirma as duas afirmações.

In [7]:
print("numero de condicao de X cru        : %.2e" % np.linalg.cond(X_tr))
print("numero de condicao de X padronizado: %.1f" % np.linalg.cond(Z))
print("maior VIF depois de padronizar     : %.1f" % np.max(vif))
print()
print("Padronizar consertou o condicionamento. A colinearidade continua inteira.")

numero de condicao de X cru        : 3.22e+09
numero de condicao de X padronizado: 28.5
maior VIF depois de padronizar     : 69.0

Padronizar consertou o condicionamento. A colinearidade continua inteira.


### O que fazer com isso

O modelo de produção é o reajustado, e a previsão dele é confiável: ela se move menos de 1%.

O que **não** é confiável é usar o coeficiente como explicação. Se a dupla escrever no relatório
que abate de bovinos tem efeito negativo sobre a produção de frango, essa frase muda de sinal na
próxima vez que o modelo for reajustado com dado novo, sem que nada tenha piorado. A Aula 10
ensinou SHAP e partial dependence justamente para não depender do coeficiente bruto, e esta é uma
demonstração de por quê.

E o MAPE que acompanha o modelo exportado continua sendo o **2,86% medido no objeto avaliado**. A
célula abaixo mostra o que aconteceria se alguém medisse o modelo de produção nos 24 meses de
teste.

In [8]:
honesto = mape(y_te, prev_avaliado)
contaminado = mape(y_te, prev_producao)

print("MAPE do modelo avaliado   nos 24 meses de teste: %.4f%%  <- o numero honesto" % honesto)
print("MAPE do modelo de producao nos 24 meses de teste: %.4f%%  <- sem significado"
      % contaminado)
print()
print("O segundo e melhor porque o modelo treinou com esses 24 meses.")
print("Publicar esse numero e o vazamento que a Aula 09 ensinou a reconhecer.")

MAPE do modelo avaliado   nos 24 meses de teste: 2.8620%  <- o numero honesto
MAPE do modelo de producao nos 24 meses de teste: 2.6492%  <- sem significado

O segundo e melhor porque o modelo treinou com esses 24 meses.
Publicar esse numero e o vazamento que a Aula 09 ensinou a reconhecer.


## 4. Exportar com `joblib`

O `Pipeline` de produção vira um arquivo. É esse arquivo que a Aula 13 carrega dentro do app
Streamlit, e ele está versionado no repositório em `app/modelo_aula12.joblib`.

In [9]:
import joblib

os.makedirs("saida", exist_ok=True)
destino = os.path.join("saida", "modelo_aula12.joblib")
joblib.dump(producao, destino)

recarregado = joblib.load(destino)
prev_recarregado = recarregado.predict(X_te)

print("arquivo: %s (%d bytes)" % (destino, os.path.getsize(destino)))
print("previsao identica apos recarregar:",
      np.array_equal(producao.predict(X_te), prev_recarregado))
print()
print("Sao %d bytes para onze coeficientes, onze medias e onze desvios." %
      os.path.getsize(destino))

arquivo: saida/modelo_aula12.joblib (1481 bytes)
previsao identica apos recarregar: True

Sao 1481 bytes para onze coeficientes, onze medias e onze desvios.


### A divergência de versão

O `joblib` grava o objeto Python inteiro, incluindo a versão do scikit-learn que o criou. Quando a
versão de quem carrega é outra, o scikit-learn emite um aviso, um por estimador do `Pipeline`.

O que a aula mediu, e vale mais que a descrição do risco: gravar com a **1.7.2** e carregar com a
**1.9.1** emite três avisos e devolve previsão **idêntica bit a bit**. O aviso avisa, e não impede
nada. O repositório tem o arquivo da 1.7.2 justamente para esta demonstração.

In [10]:
referencia = os.path.join("..", "app", "modelo_aula12_sk172.joblib")

if os.path.exists(referencia):
    with warnings.catch_warnings(record=True) as capturados:
        warnings.simplefilter("always")
        antigo = joblib.load(referencia)

    avisos = [a for a in capturados if "InconsistentVersion" in a.category.__name__]
    print("avisos emitidos: %d" % len(avisos))
    for a in avisos:
        print("  " + str(a.message).split(" This might")[0])
    print()
    print("previsao identica a do modelo atual:",
          np.array_equal(antigo.predict(X_te), producao.predict(X_te)))
else:
    print("arquivo de referencia da 1.7.2 nao encontrado (esperado fora do repositorio).")
    print("O comportamento medido na construcao da aula esta na ADR-015.")

avisos emitidos: 3
  Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.1.
  Trying to unpickle estimator LinearRegression from version 1.7.2 when using version 1.9.1.
  Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9.1.

previsao identica a do modelo atual: True


**O caminho que de fato não existe é o inverso.** Suponha que o modelo tivesse sido gravado com
scikit-learn 1.6.1 e que a dupla precisasse recriar aquele ambiente exato. No Python 3.14, o `pip`
só oferece scikit-learn a partir da **1.7.2**: não existe wheel das versões anteriores, e resta
compilar do zero.

Ou seja, a mitigação usual ("é só instalar a versão antiga") tem prazo de validade, e ele é dado
pelo interpretador, não pela biblioteca. A Aula 11 esbarrou no mesmo muro pelo outro lado, quando o
PyCaret estável recusou o Python do acervo.

O que sobra de prático é declarar as versões junto com o modelo, que é exatamente o que a próxima
seção faz.

## 5. MLflow: registrar o experimento

Um `.joblib` sozinho não diz de onde veio. Daqui a três meses, a pergunta "por que este modelo e
não outro?" precisa de resposta, e ela não está no binário.

O MLflow registra cada execução com os parâmetros, as métricas e o ambiente. A célula abaixo
registra as duas execuções desta aula.

**A aula usa um backend diferente do autoestudo.** O autoestudo "MLflow: primeiros passos" ensina o
backend de arquivos, a pasta `mlruns/`, que é o padrão histórico da ferramenta. A versão atual do
MLflow **recusa esse backend** e devolve `MlflowException: The filesystem tracking backend is in
maintenance mode`. O caminho suportado é um banco, e `sqlite:///mlflow.db` é o mais simples deles:
é um arquivo só, não exige servidor, e roda no Colab igual.

Isso é o mesmo tipo de descompasso que a Aula 11 encontrou no PyCaret, e vale como conteúdo: a
versão da biblioteca faz parte da descrição do experimento, e um tutorial de dois anos atrás
descreve um mundo que não existe mais. O motivo completo está na `ADR-015`.

In [11]:
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore")

try:
    import mlflow
except ImportError:
    print("instalando o mlflow (demora um pouco na primeira vez)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])
    import mlflow

import sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("aula12-modelo-final-frango")

for rotulo, objeto, n_treino, metrica in [
    ("avaliado", pipe, len(treino), honesto),
    ("producao", producao, len(base), None),
]:
    with mlflow.start_run(run_name=rotulo):
        mlflow.log_param("modelo", "LinearRegression")
        mlflow.log_param("escalador", "StandardScaler")
        mlflow.log_param("alvo", ALVO + " em nivel")
        mlflow.log_param("n_features", len(FEATURES))
        mlflow.log_param("meses_de_treino", n_treino)
        mlflow.log_param("corte_de_teste", teste["periodo"].iloc[0])
        mlflow.log_param("versao_sklearn", sklearn.__version__)
        mlflow.log_param("versao_dados", "SIDRA mensal ate " + base["periodo"].iloc[-1])
        if metrica is not None:
            mlflow.log_metric("mape_teste", metrica)
        else:
            mlflow.set_tag("observacao",
                           "reajustado sobre a base completa; nao ha teste fora da amostra")

print("duas execucoes registradas em mlflow.db")

2026/09/17 08:56:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/17 08:56:02 INFO mlflow.store.db.utils: Updating database tables


2026/09/17 08:56:02 INFO mlflow.tracking.fluent: Experiment with name 'aula12-modelo-final-frango' does not exist. Creating a new experiment.


duas execucoes registradas em mlflow.db


Note o que foi registrado, e principalmente o que **não** foi: a execução de produção não tem
`mape_teste`. Ela não pode ter, porque não existe conjunto de teste fora da amostra para ela. No
lugar da métrica entra uma tag dizendo isso.

É uma decisão pequena e é o coração do rastreamento honesto: o registro precisa ser capaz de dizer
"esta execução não tem essa métrica", em vez de repetir o número da outra.

A célula abaixo lê de volta o que foi registrado.

In [12]:
execucoes = mlflow.search_runs(experiment_names=["aula12-modelo-final-frango"])
colunas = [c for c in execucoes.columns
           if c.startswith("params.") or c.startswith("metrics.")
           or c == "tags.mlflow.runName"]
print(execucoes[colunas].to_string(index=False))

 metrics.mape_teste            params.alvo params.meses_de_treino      params.versao_dados params.versao_sklearn    params.modelo params.n_features params.corte_de_teste params.escalador tags.mlflow.runName
                NaN abate_frangos em nivel                    339 SIDRA mensal ate 2026-03                 1.9.1 LinearRegression                11               2024-04   StandardScaler            producao
           2.862014 abate_frangos em nivel                    315 SIDRA mensal ate 2026-03                 1.9.1 LinearRegression                11               2024-04   StandardScaler            avaliado


Para abrir a interface do MLflow na própria máquina, fora do notebook:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Ela sobe em `http://localhost:5000` e mostra as duas execuções lado a lado. O `mlflow.db` e a pasta
`mlartifacts/` não são versionados neste repositório, pelo mesmo motivo do `logs.log` do PyCaret na
Aula 11: são reescritos a cada execução e sujariam a árvore de trabalho no CI.

## 6. O que a dupla entrega na ART.8

**ART.8 Modelo Final, peso 4**, na Sprint 5.

O que sai desta aula para a entrega:

| Item | O que é |
| --- | --- |
| O `Pipeline` exportado | um arquivo, com pré-processamento e modelo juntos |
| O número que o acompanha | o MAPE medido no objeto avaliado, não no exportado |
| As versões declaradas | Python, scikit-learn e a data final da base |
| O registro no MLflow | parâmetros, métricas e o que cada execução não tem |

E a advertência que veio do achado da seção 3: **o que a dupla publica como explicação do modelo
precisa ser estável**. A previsão é. O coeficiente não é.

Na Aula 13, este mesmo `.joblib` vira a tela que a LDC vê.